# A3 – Fire escape (IFC Geometry, Spaces & Connectivity)

**Group 07 | Fire Evacuation | BIM Role: Analyst**

This notebook demonstrates how to inspect an IFC model using Python and IfcOpenShell, focusing on building elements, their properties, spaces, and possible connections.

The purpose is to understand how IFC data can support the development of a more general fire evacuation checker.

**Scope:** This tutorial focuses on model data inspection and does not verify complete evacuation routes or fire safety compliance.



## 1. Load an IFC Model

We begin by importing IfcOpenShell and opening an IFC model.
IfcOpenShell allows Python to access building elements, properties, geometry and relationships stored in IFC files.
Make sure that IfcOpenShell is installed and that the IFC file path is correct.

In [ ]:
from pathlib import Path
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util.element
import ifcopenshell.util.unit

IFC_PATH = Path("B308.ifc")

model = ifcopenshell.open(str(IFC_PATH))

print("IFC model loaded successfully")
print("IFC schema:", model.schema)


## 2. Identify Spaces, Zones, Doors and Stairs

The IFC model contains different building elements classified by IFC types.

The following classes are relevant for fire evacuation analysis:

- **IfcSpace:** Represents a room or spatial area, such as a corridor or stairwell.
- **IfcZone:** Groups spaces that share a particular function.
- **IfcDoor:** Represents doors within the building.
- **IfcStair:** Represents stair elements.
- **IfcStairFlight:** Represents individual stair flights.

The code extracts these elements and displays their names and GlobalIds.

In [ ]:

classes = [
    "IfcSpace",
    "IfcDoor",
    "IfcStair",
    "IfcStairFlight",
    "IfcBuildingStorey",
    "IfcRelSpaceBoundary"
]

for ifc_class in classes:
    objects = model.by_type(ifc_class)
    print(ifc_class, len(objects))



##  3. Check the FireExit Property

Next, we investigate whether doors contain information identifying emergency exits.

The relevant IFC property is:

`Pset_DoorCommon.FireExit`

This property can help identify doors designated as fire exits.

The following code extracts the FireExit value from the doors in the IFC model.



In [ ]:

doors = model.by_type("IfcDoor")

for door in doors:

    fire_exit = get_pset(
        door,
        "Pset_DoorCommon",
        "FireExit"
    )

    print(
        door.GlobalId,
        door.Name,
        "FireExit =",
        fire_exit
    )



### Understanding FireExit Values

The FireExit property can return three different values:

- **True:** The door is explicitly identified as a fire exit.
- **False:** The door is explicitly identified as not being a fire exit.
- **None:** The information is missing.

A door should not automatically be identified as an emergency exit simply because it is located on an external wall, has a certain width or is close to the outside.

These assumptions may be incorrect.

Therefore, missing information should be reported rather than assumed.



## 4. Check Door Widths

Next, we investigate the dimensions of the doors in the IFC model.

The `OverallWidth` attribute represents the nominal width of a door.

Door widths are relevant for fire evacuation because they can help assess whether doors meet specific design requirements.

In the Building 308 report, a width of 1.20 m is mentioned for the main entrance and certain public spaces.

However, nominal width is not necessarily the same as the effective clear passage width.

The following code extracts the recorded door widths and identifies doors with a nominal width of at least 1.20 m.


In [ ]:

scale = ifcopenshell.util.unit.calculate_unit_scale(model)

wide_doors = 0
narrow_doors = 0
missing_width = 0

for door in doors:

    width = door.OverallWidth

    if width is None or width <= 0:
        missing_width += 1
        continue

    width_m = width * scale

    if width_m >= 1.20:
        wide_doors += 1
    else:
        narrow_doors += 1

print("Doors with nominal width >= 1.20 m:", wide_doors)
print("Doors with nominal width < 1.20 m:", narrow_doors)
print("Doors without recorded width:", missing_width)



### Understanding Door Widths

The code divides the doors into three categories:

- **Width ≥ 1.20 m:** Doors with a recorded nominal width of at least 1.20 m.
- **Width < 1.20 m:** Doors with a smaller nominal width.
- **Missing width:** Doors without a valid recorded width.

The code converts the dimensions into metres using the IFC model's unit scale.

This is a preliminary screening of door dimensions.

It does not verify whether a door meets fire safety requirements, since effective clear widths may differ from nominal widths and the 1.20 m requirement does not apply to every door.



## 5. Check Space Boundaries

Next, we investigate whether the IFC model contains information about the relationships between spaces and building elements.

One relevant IFC relationship is:

`IfcRelSpaceBoundary`

It describes how a space is bounded by elements such as walls or doors.

This information can help identify possible connections between spaces and building elements.

However, space boundary relationships alone do not prove that a complete evacuation route exists.


In [ ]:

boundaries = model.by_type("IfcRelSpaceBoundary")

print("Total space boundaries:", len(boundaries))


## 6. Check Whether Boundaries Reference Doors

If the model contains space boundaries, we can investigate whether any of them reference doors.

This can help identify relationships between spaces and doors.

The following code counts how many space boundary relationships reference an IfcDoor element.

A recorded relationship does not necessarily mean that the door provides a usable connection between two spaces.


In [ ]:

door_boundaries = 0

for rel in boundaries:

    element = getattr(
        rel,
        "RelatedBuildingElement",
        None
    )

    if (
        element is not None
        and element.is_a("IfcDoor")
    ):

        door_boundaries += 1


print(
    "Space boundaries related to doors:",
    door_boundaries)



## 7. Create an Information-Readiness Checker

We can now combine the previous checks into one Python function.

The function investigates whether the IFC model contains:

- IfcSpace objects.
- IfcDoor objects.
- IfcRelSpaceBoundary relationships.
- Space boundaries referencing doors.
- Doors with FireExit information.

The checker returns one of two results:

**READY:** The model passes the basic information checks defined in this tutorial.

**INSUFFICIENT DATA:** One or more required information checks are not satisfied.

A READY result does not mean that the building has passed a complete fire evacuation analysis.


In [ ]:

def check_egress_readiness(model):

    spaces = model.by_type("IfcSpace")
    doors = model.by_type("IfcDoor")
    boundaries = model.by_type("IfcRelSpaceBoundary")

    exit_count = 0
    missing_fire_exit = 0
    door_boundary_count = 0

    for door in doors:

        value = get_pset(
            door,
            "Pset_DoorCommon",
            "FireExit"
        )

        if value is True:
            exit_count += 1

        elif value is None:
            missing_fire_exit += 1

    for rel in boundaries:

        element = getattr(
            rel,
            "RelatedBuildingElement",
            None
        )

        if (
            element is not None
            and element.is_a("IfcDoor")
        ):
            door_boundary_count += 1

    issues = []

    if not spaces:
        issues.append("No IfcSpace objects")

    if not doors:
        issues.append("No IfcDoor objects")

    if not boundaries:
        issues.append(
            "No IfcRelSpaceBoundary relationships"
        )

    if boundaries and door_boundary_count == 0:
        issues.append(
            "No space boundaries related to doors"
        )

    if exit_count == 0:
        issues.append(
            "No explicit FireExit=True door"
        )

    if missing_fire_exit > 0:
        issues.append(
            f"{missing_fire_exit} doors have no FireExit value"
        )

    if issues:
        status = "INSUFFICIENT DATA"
    else:
        status = "READY"

    return status, issues



## 8. Run the Readiness Check

We can now apply the function to the IFC model.

The program will display the information-readiness status and list any missing information identified by the checker.


In [ ]:

status, issues = check_egress_readiness(model)

print("STATUS:", status)

for issue in issues:
    print("-", issue)


## 9. Connection to IDS

The information identified in this tutorial can also be described as information requirements.

| Required Information | IFC Location | Purpose |
|---|---|---|
| Spaces | IfcSpace | Spatial areas for evacuation analysis |
| Doors | IfcDoor | Transitions between areas |
| Fire exits | Pset_DoorCommon.FireExit | Identify designated emergency exits |
| Space boundaries | IfcRelSpaceBoundary | Relate spaces to building elements |
| Storeys | IfcBuildingStorey | Multi-storey context |
| Stairs | IfcStair / IfcStairFlight | Vertical circulation |
| Door widths | IfcDoor.OverallWidth | Check nominal door dimensions |

An Information Delivery Specification (IDS) can be used to define and validate required IFC information.

Python can then be used for more complex analysis.

The workflow becomes:

    Define information requirements
                ↓
       Check IFC information
                ↓
    Is enough information available?
           ↙           ↘
         NO             YES
          ↓              ↓
    Report missing    Continue with
     information      further analysis



## 10. Conclusion

This tutorial demonstrates how Python and IfcOpenShell can be used to investigate fire-egress information in an IFC model.

The analysis includes:

- Identifying relevant IFC elements.
- Checking FireExit properties.
- Extracting and examining door widths.
- Investigating space boundary relationships.
- Identifying missing information.

These methods provide a starting point for developing a more general IFC fire evacuation checker.

However, the code does not verify complete evacuation routes, calculate evacuation distances or automatically determine whether spaces are connected.

**Missing IFC information should be reported rather than interpreted as a fire safety failure.**
